# Regional GAA Profiling
Owner: Anje
Table: `pondolens.dev_bronze.regional_expenditure_gaa`
Baseline: 1,920 rows

## Goal
Identify data issues and propose Silver treatments. Do not modify Bronze.

## Checks
- Check counts by file/year, column types, nulls/blanks, and `_rescued_data`.
- Define what one row represents; check exact duplicates and repeated business keys.
- Verify regions, years, categories, particulars, numeric amounts, units, and price basis.
- Identify totals/subtotals to avoid double counting.
- Compare sample amounts and totals with the original source.
- With Joy, check whether GAA and obligations have matching scope, periods, and regions.
- Check regional compatibility with population and GRDP.

## Findings
| Check / Issue | Affected rows / Evidence | Proposed Silver treatment |
|---|---|---|
| Dashes in expenditure_amount column| 336 rows in 2025 table, 344 rows in 2024 table| treat as NULL since no data was inputted in the original tables|
| Special characters in particulars column (). The data contains both the department and the acronym of the department inside the () | 40 distinct column_value with 20 occurences each for both 2024 and 2025 tables. 39 of these values are the agency names while the other value is GAA(updated)| separate the acronym from the full agency names|
| when particulars = 'All', these are aggregate rows. Other particulars are department or fund detail rows rathen than automatically being subtotals | All rows| Keep all rows in Silver and add two classification columns. This preserves published totals for validation while allowing dashboards to select detail rows without double counting. financial_row_type: grand_total, category_total, detail AND geographic_row_type: geographic_total, nonregional, regional|
|Expenditure amount in thousand pesos|All rows| convert expenditure amount to be in thousand pesos and remove unit_amoount column|

## Assumptions to Verify
| Interpretation / Question | Evidence | Status |
|---|---|---|
| To be completed | | Pending |

## Semantic Notes
Document the confirmed spending measure, unit conversion, and rows safe to sum.
Do not label GAA amounts as actual spending without source confirmation.

## Conclusion
- Required Silver changes:
- Ready for Q1/Q3? Why / why not:
- Open questions:

In [0]:
%sql
SELECT *
FROM pondolens.dev_bronze.regional_expenditure_gaa
LIMIT 10;

In [0]:
%sql
DESCRIBE TABLE pondolens.dev_bronze.regional_expenditure_gaa;

In [0]:
%sql
-- List of all regions

SELECT
    DISTINCT(regions) as region_list
FROM pondolens.dev_bronze.regional_expenditure_gaa

In [0]:
%sql
-- Null counts
SELECT
    COUNT(*) as total_rows,
    COUNT_IF(source_name IS NULL) as source_name_nulls,
    COUNT_IF(source_page IS NULL) as source_page_nulls,
    COUNT_IF(price_basis IS NULL) as price_basis_nulls,
    COUNT_IF(year IS NULL) as year_nulls,
    COUNT_IF(category IS NULL) as category_nulls,
    COUNT_IF(particulars IS NULL) as particulars_nulls,
    COUNT_IF(regions IS NULL) as regions_nulls,
    COUNT_IF(expenditure_amount IS NULL) as expenditure_amount_nulls,
    COUNT_IF(unit_amount IS NULL) as unit_amount_nulls,
    COUNT_IF(_rescued_data IS NULL) as _rescued_data_nulls,
    COUNT_IF(_ingested_at IS NULL) as ingestion_timestamp_nulls,
    COUNT_IF(_source_file IS NULL) as source_file_nulls
FROM
    pondolens.dev_bronze.regional_expenditure_gaa

In [0]:
%sql
-- Duplicates
SELECT
    source_name,
    source_page,
    price_basis,
    year,
    category,
    particulars,
    regions,
    expenditure_amount,
    unit_amount,
    COUNT(*) AS duplicate_count
FROM pondolens.dev_bronze.regional_expenditure_gaa
GROUP BY
    source_name,
    source_page,
    price_basis,
    year,
    category,
    particulars,
    regions,
    expenditure_amount,
    unit_amount
HAVING COUNT(*) > 1;

In [0]:
%sql
-- Check for special characters 

WITH column_values AS (
    SELECT
        source_name AS file_name,
        STACK(
            9,
            'source_name', CAST(source_name AS STRING),
            'source_page', CAST(source_page AS STRING),
            'price_basis', CAST(price_basis AS STRING),
            'year', CAST(year AS STRING),
            'category', CAST(category AS STRING),
            'particulars', CAST(particulars AS STRING),
            'regions', CAST(regions AS STRING),
            'expenditure_amount', CAST(expenditure_amount AS STRING),
            'unit_amount', CAST(unit_amount AS STRING)
        ) AS (column_name, column_value)
    FROM pondolens.dev_bronze.regional_expenditure_gaa
)
SELECT
    file_name,
    column_name,
    column_value,
    COUNT(*) AS occurrences
FROM column_values
WHERE column_value RLIKE '[^a-zA-Z0-9 ]'
GROUP BY file_name, column_name, column_value
ORDER BY column_name, occurrences DESC;

In [0]:
%sql
SELECT
    *
FROM pondolens.raw.regional_expenditure_2024
WHERE particulars <> 'All'
LIMIT 10;